# Visual Search System - Model Development

This notebook covers the model architecture and training strategies for building a visual search system. We'll explore CNN and Transformer-based architectures, contrastive learning techniques, and how to construct training datasets from user interactions.

## Learning Objectives
- Understand why neural networks are ideal for visual search
- Compare CNN (ResNet) vs Transformer (ViT) architectures
- Learn contrastive training techniques (SimCLR, MoCo)
- Construct training datasets from user interactions
- Implement loss functions for embedding learning

## 1. Why Neural Networks for Visual Search?

Neural networks are the preferred choice for visual search systems because they can:

1. **Handle Unstructured Data**: Images are high-dimensional unstructured data that traditional ML models struggle with
2. **Learn Hierarchical Features**: CNNs learn features from edges → textures → parts → objects
3. **Produce Dense Embeddings**: The output can be a compact vector representation of the image
4. **Transfer Learning**: Pre-trained models on ImageNet can be fine-tuned for visual search

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple

# Visualization of embedding concept
def visualize_embedding_space():
    """Visualize how images are mapped to embedding space"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: Image space (high dimensional)
    ax1 = axes[0]
    ax1.text(0.5, 0.5, 'Image Space\n(224×224×3 = 150,528 dims)', 
             ha='center', va='center', fontsize=14,
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    ax1.set_title('Original Image Space', fontsize=12)
    ax1.axis('off')
    
    # Right: Embedding space (low dimensional)
    ax2 = axes[1]
    
    # Generate sample embeddings for different categories
    np.random.seed(42)
    categories = ['Dogs', 'Cats', 'Cars', 'Flowers']
    colors = ['blue', 'orange', 'green', 'red']
    
    for i, (cat, color) in enumerate(zip(categories, colors)):
        center = np.array([np.cos(i * np.pi/2), np.sin(i * np.pi/2)]) * 0.5
        points = center + np.random.randn(10, 2) * 0.1
        ax2.scatter(points[:, 0], points[:, 1], c=color, label=cat, s=50, alpha=0.7)
    
    ax2.set_title('Embedding Space (e.g., 128 dims)', fontsize=12)
    ax2.set_xlabel('Dimension 1')
    ax2.set_ylabel('Dimension 2')
    ax2.legend()
    ax2.set_xlim(-1, 1)
    ax2.set_ylim(-1, 1)
    
    # Add arrow between plots
    fig.text(0.5, 0.5, '→ Neural Network →', ha='center', va='center', 
             fontsize=16, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

visualize_embedding_space()

## 2. Model Architecture Options

### 2.1 Convolutional Neural Networks (CNNs)

CNNs like ResNet are the traditional choice for image understanding:

**Architecture Flow:**
```
Input Image (224×224×3)
    ↓
Convolutional Layers (feature extraction)
    ↓
Global Average Pooling
    ↓
Fully Connected Layer
    ↓
Embedding Vector (128 or 256 dims)
```

**Advantages:**
- Well-understood architecture
- Efficient inference
- Translation invariance built-in
- Many pre-trained models available

In [ ]:
# Simplified CNN architecture for embedding generation
# Using PyTorch-like pseudocode

class SimplifiedCNNEmbedding:
    """
    Conceptual CNN architecture for visual search embeddings.
    This is pseudocode to illustrate the architecture.
    """
    def __init__(self, embedding_dim=128):
        self.architecture = {
            'input': '224x224x3 RGB image',
            'conv_block_1': {
                'conv': '7x7, 64 filters, stride 2',
                'batch_norm': True,
                'activation': 'ReLU',
                'max_pool': '3x3, stride 2',
                'output_shape': '56x56x64'
            },
            'conv_block_2': {
                'residual_blocks': 3,
                'filters': 64,
                'output_shape': '56x56x64'
            },
            'conv_block_3': {
                'residual_blocks': 4,
                'filters': 128,
                'output_shape': '28x28x128'
            },
            'conv_block_4': {
                'residual_blocks': 6,
                'filters': 256,
                'output_shape': '14x14x256'
            },
            'conv_block_5': {
                'residual_blocks': 3,
                'filters': 512,
                'output_shape': '7x7x512'
            },
            'global_avg_pool': {
                'output_shape': '512'
            },
            'fc_layer': {
                'units': embedding_dim,
                'output_shape': str(embedding_dim)
            },
            'l2_normalize': {
                'description': 'Normalize to unit sphere',
                'output_shape': str(embedding_dim)
            }
        }
    
    def describe(self):
        print("ResNet-based Embedding Architecture:")
        print("=" * 50)
        for name, config in self.architecture.items():
            if isinstance(config, dict):
                print(f"\n{name}:")
                for k, v in config.items():
                    print(f"  {k}: {v}")
            else:
                print(f"\n{name}: {config}")

model = SimplifiedCNNEmbedding()
model.describe()

### 2.2 Vision Transformers (ViT)

Transformers have recently shown excellent results for image understanding:

**Architecture Flow:**
```
Input Image (224×224×3)
    ↓
Split into Patches (16×16 each → 196 patches)
    ↓
Patch Embedding + Position Embedding
    ↓
Transformer Encoder (12 layers)
    ↓
[CLS] Token Output
    ↓
Embedding Vector (768 dims)
```

**Advantages:**
- Captures global context from the start
- Scales well with more data
- Flexible attention patterns
- State-of-the-art performance on many benchmarks

In [ ]:
def visualize_vit_architecture():
    """Visualize Vision Transformer patch processing"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Left: Original image with patches
    ax1 = axes[0]
    img = np.random.rand(224, 224, 3)
    ax1.imshow(img)
    
    # Draw patch grid
    patch_size = 16
    for i in range(0, 224, patch_size):
        ax1.axhline(y=i, color='red', linewidth=0.5)
        ax1.axvline(x=i, color='red', linewidth=0.5)
    
    ax1.set_title('Image Split into 16×16 Patches\n(14×14 = 196 patches)', fontsize=10)
    ax1.axis('off')
    
    # Middle: Patch embeddings
    ax2 = axes[1]
    patch_embeddings = np.random.rand(14, 14)
    im = ax2.imshow(patch_embeddings, cmap='viridis')
    ax2.set_title('Patch Embeddings\n(Each patch → 768-dim vector)', fontsize=10)
    ax2.axis('off')
    
    # Right: Transformer output
    ax3 = axes[2]
    # Show attention pattern
    attention = np.random.rand(196, 196)
    attention = (attention + attention.T) / 2  # Symmetric
    ax3.imshow(attention, cmap='hot')
    ax3.set_title('Self-Attention Pattern\n(Global context)', fontsize=10)
    ax3.set_xlabel('Patches')
    ax3.set_ylabel('Patches')
    
    plt.tight_layout()
    plt.show()

visualize_vit_architecture()

### 2.3 Architecture Comparison

| Aspect | CNN (ResNet-50) | ViT-Base |
|--------|-----------------|----------|
| Parameters | ~25M | ~86M |
| Embedding Dim | 2048 (can reduce) | 768 |
| Inference Speed | Faster | Slower |
| Data Requirements | Works with less data | Needs more data |
| Accuracy (ImageNet) | 76-79% | 77-81% |
| Best For | Production, limited data | Maximum accuracy, large data |

## 3. Contrastive Training

The key to learning good visual embeddings is **contrastive learning**. The goal is to train the model so that:
- **Similar images** have embeddings that are **close** in the embedding space
- **Dissimilar images** have embeddings that are **far apart**

### 3.1 Training Data Construction

We need triplets or pairs of images:
1. **Query Image**: The image we're searching with
2. **Positive Image**: A visually similar image (should be close)
3. **Negative Images**: Dissimilar images (should be far)

In [ ]:
import pandas as pd

# Example: Constructing training data from user interactions
user_interactions = pd.DataFrame({
    'user_id': ['u1', 'u1', 'u1', 'u1', 'u2', 'u2', 'u2'],
    'query_image_id': ['img_001', 'img_001', 'img_001', 'img_001', 'img_002', 'img_002', 'img_002'],
    'displayed_image_id': ['img_101', 'img_102', 'img_103', 'img_104', 'img_201', 'img_202', 'img_203'],
    'position': [1, 2, 3, 4, 1, 2, 3],
    'interaction_type': ['click', 'impression', 'impression', 'click', 'impression', 'click', 'impression']
})

print("User Interaction Data:")
print(user_interactions)
print("\n" + "="*60)

# Extract positive and negative pairs
def extract_training_pairs(interactions):
    training_samples = []
    
    for query_img in interactions['query_image_id'].unique():
        query_data = interactions[interactions['query_image_id'] == query_img]
        
        # Clicks are positive examples
        positives = query_data[query_data['interaction_type'] == 'click']['displayed_image_id'].tolist()
        
        # Impressions without clicks are negative examples
        negatives = query_data[query_data['interaction_type'] == 'impression']['displayed_image_id'].tolist()
        
        training_samples.append({
            'query': query_img,
            'positives': positives,
            'negatives': negatives
        })
    
    return training_samples

training_data = extract_training_pairs(user_interactions)
print("\nExtracted Training Pairs:")
for sample in training_data:
    print(f"  Query: {sample['query']}")
    print(f"    Positives (clicked): {sample['positives']}")
    print(f"    Negatives (not clicked): {sample['negatives']}")
    print()

### 3.2 Sources of Training Data

| Method | Pros | Cons |
|--------|------|------|
| **Human Judgments** | High accuracy, explicit labels | Expensive, slow, doesn't scale |
| **User Clicks** | Free, abundant, implicit feedback | Noisy, sparse, position bias |
| **Self-Supervision** | No labels needed, infinite data | May not align with search intent |

In practice, we often combine these approaches:
1. Pre-train with self-supervision (SimCLR, MoCo)
2. Fine-tune with user clicks

## 4. Self-Supervised Learning

Self-supervised methods create training signals without manual labels by using data augmentation.

### 4.1 SimCLR (Simple Contrastive Learning of Representations)

**Core Idea:** Different augmentations of the same image should have similar embeddings.

**Algorithm:**
1. Take an image and create two random augmentations
2. These two views form a positive pair
3. All other images in the batch are negatives
4. Train to maximize agreement between positive pairs

In [ ]:
def visualize_simclr_augmentations():
    """Visualize SimCLR data augmentation pipeline"""
    fig, axes = plt.subplots(2, 4, figsize=(14, 6))
    
    # Original image (simulated)
    np.random.seed(42)
    original = np.random.rand(100, 100, 3)
    
    # First row: Augmentation types
    augmentations = [
        ('Original', original),
        ('Random Crop', original[10:90, 10:90]),
        ('Color Jitter', np.clip(original * np.random.uniform(0.8, 1.2, 3), 0, 1)),
        ('Grayscale', np.stack([original.mean(axis=2)]*3, axis=2))
    ]
    
    for i, (name, img) in enumerate(augmentations):
        axes[0, i].imshow(img)
        axes[0, i].set_title(name, fontsize=10)
        axes[0, i].axis('off')
    
    # Second row: SimCLR positive pairs
    axes[1, 0].text(0.5, 0.5, 'Same Image\nDifferent Augmentations', 
                    ha='center', va='center', fontsize=12,
                    transform=axes[1, 0].transAxes)
    axes[1, 0].axis('off')
    
    # View 1
    view1 = np.clip(original[5:95, 5:95] * 1.1, 0, 1)
    axes[1, 1].imshow(view1)
    axes[1, 1].set_title('View 1 (Positive)', fontsize=10)
    axes[1, 1].axis('off')
    
    # View 2
    view2 = np.clip(original[10:90, 15:95] * 0.9, 0, 1)
    axes[1, 2].imshow(view2)
    axes[1, 2].set_title('View 2 (Positive)', fontsize=10)
    axes[1, 2].axis('off')
    
    # Different image (negative)
    negative = np.random.rand(80, 80, 3)
    axes[1, 3].imshow(negative)
    axes[1, 3].set_title('Other Image (Negative)', fontsize=10)
    axes[1, 3].axis('off')
    
    plt.suptitle('SimCLR Augmentation Pipeline', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_simclr_augmentations()

In [ ]:
# SimCLR augmentation pipeline (pseudocode)
def simclr_augmentation_pipeline():
    """
    SimCLR augmentation pipeline configuration.
    Returns a description of the augmentation steps.
    """
    pipeline = [
        {
            'name': 'RandomResizedCrop',
            'params': {'scale': (0.08, 1.0), 'ratio': (0.75, 1.33)},
            'purpose': 'Crop random portion and resize to 224x224'
        },
        {
            'name': 'RandomHorizontalFlip',
            'params': {'p': 0.5},
            'purpose': 'Flip image horizontally with 50% probability'
        },
        {
            'name': 'ColorJitter',
            'params': {'brightness': 0.8, 'contrast': 0.8, 'saturation': 0.8, 'hue': 0.2},
            'purpose': 'Random color distortions'
        },
        {
            'name': 'RandomGrayscale',
            'params': {'p': 0.2},
            'purpose': 'Convert to grayscale with 20% probability'
        },
        {
            'name': 'GaussianBlur',
            'params': {'kernel_size': 23, 'sigma': (0.1, 2.0)},
            'purpose': 'Apply random Gaussian blur'
        },
        {
            'name': 'Normalize',
            'params': {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]},
            'purpose': 'ImageNet normalization'
        }
    ]
    
    print("SimCLR Augmentation Pipeline:")
    print("=" * 60)
    for i, step in enumerate(pipeline, 1):
        print(f"\n{i}. {step['name']}")
        print(f"   Parameters: {step['params']}")
        print(f"   Purpose: {step['purpose']}")
    
    return pipeline

pipeline = simclr_augmentation_pipeline()

### 4.2 MoCo (Momentum Contrast)

**Core Idea:** Maintain a large dictionary of negative samples using a momentum encoder.

**Advantages over SimCLR:**
- Can use more negatives without increasing batch size
- More memory efficient
- Momentum encoder provides consistent representations

In [ ]:
def visualize_moco_architecture():
    """Visualize MoCo architecture"""
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    
    # Draw components
    boxes = [
        {'name': 'Query\nImage', 'pos': (0.1, 0.7), 'color': 'lightblue'},
        {'name': 'Key\nImage', 'pos': (0.1, 0.3), 'color': 'lightgreen'},
        {'name': 'Query\nEncoder', 'pos': (0.35, 0.7), 'color': 'lightblue'},
        {'name': 'Momentum\nEncoder', 'pos': (0.35, 0.3), 'color': 'lightgreen'},
        {'name': 'Query\nEmbedding q', 'pos': (0.6, 0.7), 'color': 'lightblue'},
        {'name': 'Key\nEmbedding k', 'pos': (0.6, 0.3), 'color': 'lightgreen'},
        {'name': 'Queue\n(negatives)', 'pos': (0.6, 0.1), 'color': 'lightyellow'},
        {'name': 'Contrastive\nLoss', 'pos': (0.85, 0.5), 'color': 'lightcoral'},
    ]
    
    for box in boxes:
        ax.add_patch(plt.Rectangle(
            (box['pos'][0]-0.08, box['pos'][1]-0.08), 0.16, 0.16,
            facecolor=box['color'], edgecolor='black', linewidth=2
        ))
        ax.text(box['pos'][0], box['pos'][1], box['name'],
                ha='center', va='center', fontsize=9, fontweight='bold')
    
    # Draw arrows
    arrows = [
        ((0.18, 0.7), (0.27, 0.7)),   # Query to encoder
        ((0.18, 0.3), (0.27, 0.3)),   # Key to momentum encoder
        ((0.43, 0.7), (0.52, 0.7)),   # Encoder to embedding
        ((0.43, 0.3), (0.52, 0.3)),   # Mom encoder to embedding
        ((0.68, 0.7), (0.77, 0.55)),  # q to loss
        ((0.68, 0.3), (0.77, 0.45)),  # k to loss
        ((0.68, 0.15), (0.77, 0.42)), # Queue to loss
    ]
    
    for start, end in arrows:
        ax.annotate('', xy=end, xytext=start,
                   arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    
    # Momentum update arrow
    ax.annotate('', xy=(0.35, 0.54), xytext=(0.35, 0.62),
               arrowprops=dict(arrowstyle='->', color='blue', lw=2, ls='--'))
    ax.text(0.42, 0.58, 'Momentum\nUpdate', fontsize=8, color='blue')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 0.9)
    ax.axis('off')
    ax.set_title('MoCo (Momentum Contrast) Architecture', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_moco_architecture()

## 5. Loss Functions for Embedding Learning

### 5.1 Contrastive Loss (InfoNCE)

The most common loss function for contrastive learning:

$$\mathcal{L} = -\log \frac{\exp(\text{sim}(q, k^+) / \tau)}{\sum_{i=0}^{K} \exp(\text{sim}(q, k_i) / \tau)}$$

Where:
- $q$ = query embedding
- $k^+$ = positive key embedding
- $k_i$ = all key embeddings (positive + negatives)
- $\tau$ = temperature parameter
- $\text{sim}$ = similarity function (usually cosine similarity or dot product)

In [ ]:
def compute_contrastive_loss(query_emb, positive_emb, negative_embs, temperature=0.07):
    """
    Compute InfoNCE contrastive loss.
    
    Args:
        query_emb: Query embedding (1, dim)
        positive_emb: Positive embedding (1, dim)
        negative_embs: Negative embeddings (n_neg, dim)
        temperature: Temperature for softmax
    
    Returns:
        Contrastive loss value
    """
    # Normalize embeddings
    query_emb = query_emb / np.linalg.norm(query_emb)
    positive_emb = positive_emb / np.linalg.norm(positive_emb)
    negative_embs = negative_embs / np.linalg.norm(negative_embs, axis=1, keepdims=True)
    
    # Compute similarities
    pos_sim = np.dot(query_emb, positive_emb) / temperature
    neg_sims = np.dot(negative_embs, query_emb) / temperature
    
    # Compute loss
    all_sims = np.concatenate([[pos_sim], neg_sims])
    exp_sims = np.exp(all_sims - np.max(all_sims))  # Numerical stability
    loss = -np.log(exp_sims[0] / np.sum(exp_sims))
    
    return loss, pos_sim * temperature, neg_sims * temperature

# Example
np.random.seed(42)
query = np.random.randn(128)
positive = query + np.random.randn(128) * 0.1  # Similar to query
negatives = np.random.randn(10, 128)  # Random negatives

loss, pos_sim, neg_sims = compute_contrastive_loss(query, positive, negatives)

print(f"Contrastive Loss: {loss:.4f}")
print(f"\nPositive similarity: {pos_sim:.4f}")
print(f"Negative similarities: min={neg_sims.min():.4f}, max={neg_sims.max():.4f}, mean={neg_sims.mean():.4f}")
print(f"\nThe positive sample has much higher similarity (good!)")

### 5.2 Temperature Parameter

The temperature $\tau$ controls how "peaky" the softmax distribution is:
- **Low temperature (e.g., 0.07)**: Sharp distribution, focuses on hard negatives
- **High temperature (e.g., 1.0)**: Soft distribution, treats all negatives more equally

In [ ]:
def visualize_temperature_effect():
    """Visualize effect of temperature on softmax distribution"""
    similarities = np.array([0.9, 0.5, 0.3, 0.2, 0.1, -0.1, -0.2])
    temperatures = [0.07, 0.2, 0.5, 1.0]
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    for ax, temp in zip(axes, temperatures):
        logits = similarities / temp
        probs = np.exp(logits - np.max(logits))
        probs = probs / probs.sum()
        
        colors = ['green'] + ['red'] * (len(similarities) - 1)
        ax.bar(range(len(probs)), probs, color=colors, edgecolor='black')
        ax.set_title(f'Temperature = {temp}', fontsize=12)
        ax.set_xlabel('Sample Index')
        ax.set_ylabel('Probability')
        ax.set_ylim(0, 1)
        ax.set_xticks(range(len(probs)))
        ax.set_xticklabels(['Pos'] + [f'Neg{i}' for i in range(1, len(probs))], rotation=45)
    
    plt.suptitle('Effect of Temperature on Softmax Distribution\n(Green = Positive, Red = Negative)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_temperature_effect()

## 6. Training Pipeline

### 6.1 Complete Training Loop

In [ ]:
def training_pipeline_pseudocode():
    """
    Pseudocode for visual search model training pipeline.
    """
    pipeline = """
    VISUAL SEARCH EMBEDDING MODEL TRAINING PIPELINE
    ================================================
    
    1. DATA PREPARATION
       ├── Load images and user interaction logs
       ├── Construct positive pairs from clicked results
       ├── Sample negative pairs from impressions without clicks
       └── Apply data augmentation for self-supervision
    
    2. MODEL INITIALIZATION
       ├── Load pre-trained backbone (ResNet-50 or ViT)
       ├── Replace classification head with embedding layer
       ├── Add L2 normalization layer
       └── Initialize optimizer (Adam, lr=1e-4)
    
    3. TRAINING LOOP (for each epoch):
       │
       ├── For each batch:
       │   ├── Load query images, positive images, negative images
       │   ├── Apply augmentations
       │   ├── Forward pass: compute embeddings
       │   ├── Compute contrastive loss
       │   ├── Backward pass: compute gradients
       │   └── Update weights
       │
       ├── Validation:
       │   ├── Compute embeddings for validation set
       │   ├── Calculate recall@k on held-out queries
       │   └── Log metrics
       │
       └── Checkpoint:
           ├── Save model if validation improves
           └── Early stopping if no improvement
    
    4. POST-TRAINING
       ├── Export model for inference
       ├── Compute embeddings for all images in index
       └── Build approximate nearest neighbor index
    
    HYPERPARAMETERS:
    ─────────────────
    • Embedding dimension: 128 or 256
    • Batch size: 256-1024
    • Temperature: 0.07
    • Learning rate: 1e-4 (with cosine annealing)
    • Number of negatives: 255 (per positive)
    • Training epochs: 100-200 (with early stopping)
    """
    print(pipeline)

training_pipeline_pseudocode()

### 6.2 Fine-tuning Pre-trained Models

Starting from pre-trained models (ImageNet) significantly accelerates training:

In [ ]:
def finetuning_strategy():
    """
    Fine-tuning strategy for visual search models.
    """
    strategies = [
        {
            'phase': 'Phase 1: Feature Extraction',
            'epochs': '5-10',
            'frozen_layers': 'All backbone layers',
            'trainable_layers': 'Only embedding head',
            'learning_rate': '1e-3',
            'purpose': 'Learn embedding projection'
        },
        {
            'phase': 'Phase 2: Partial Fine-tuning',
            'epochs': '10-20',
            'frozen_layers': 'Early conv layers (1-3)',
            'trainable_layers': 'Later layers + head',
            'learning_rate': '1e-4',
            'purpose': 'Adapt high-level features'
        },
        {
            'phase': 'Phase 3: Full Fine-tuning',
            'epochs': '20-50',
            'frozen_layers': 'None',
            'trainable_layers': 'All layers',
            'learning_rate': '1e-5',
            'purpose': 'End-to-end optimization'
        }
    ]
    
    print("Fine-tuning Strategy for Visual Search Models")
    print("=" * 60)
    
    for strategy in strategies:
        print(f"\n{strategy['phase']}")
        print("-" * 40)
        print(f"  Epochs: {strategy['epochs']}")
        print(f"  Frozen: {strategy['frozen_layers']}")
        print(f"  Trainable: {strategy['trainable_layers']}")
        print(f"  Learning Rate: {strategy['learning_rate']}")
        print(f"  Purpose: {strategy['purpose']}")

finetuning_strategy()

## 7. Summary

### Key Takeaways

1. **Architecture Choice**:
   - CNNs (ResNet) for efficiency and when data is limited
   - ViT for maximum accuracy with large datasets

2. **Training Data**:
   - User clicks provide implicit positive labels
   - Self-supervision (SimCLR, MoCo) for pre-training
   - Fine-tuning on domain-specific data

3. **Contrastive Learning**:
   - Pull positive pairs together
   - Push negative pairs apart
   - Temperature controls the sharpness of the distribution

4. **Training Strategy**:
   - Start with pre-trained models
   - Progressive unfreezing for stable training
   - Monitor embedding quality with recall@k

In [ ]:
# Summary diagram
def model_development_summary():
    summary = """
    ╔══════════════════════════════════════════════════════════════╗
    ║          VISUAL SEARCH MODEL DEVELOPMENT SUMMARY             ║
    ╠══════════════════════════════════════════════════════════════╣
    ║                                                              ║
    ║   Architecture Options:                                      ║
    ║   ├── CNN (ResNet-50): Efficient, well-understood           ║
    ║   └── ViT: State-of-the-art, needs more data                ║
    ║                                                              ║
    ║   Training Approaches:                                       ║
    ║   ├── Supervised: Human labels (expensive)                  ║
    ║   ├── Weakly Supervised: User clicks (noisy)                ║
    ║   └── Self-Supervised: SimCLR/MoCo (scalable)               ║
    ║                                                              ║
    ║   Loss Function: InfoNCE Contrastive Loss                   ║
    ║   ├── Positive pairs: maximize similarity                   ║
    ║   └── Negative pairs: minimize similarity                   ║
    ║                                                              ║
    ║   Output: Normalized embedding vectors (128-256 dims)       ║
    ║                                                              ║
    ╚══════════════════════════════════════════════════════════════╝
    """
    print(summary)

model_development_summary()